# 01 — Benchmark: do our data match CDC's own numbers?

**Goal.** Show that the Parquet files we built contain exactly the births and infant deaths that CDC published. If they match, every later result rests on correct data.

**Thesis part.** Step 1 (data build), before Part 1 (prediction) and Part 2 (association). See `plan/analysis_plan.md`, section 5.

**Inputs.** `data/parquet/cohort_<year>.parquet` (built by `src/build_parquet.py`), and the CDC user guides in `docs/guides/`.

**Outputs.** `reference/benchmark_counts.csv`, `reference/benchmark_table2.csv`.

**How to run.** Build the Parquet files first (`python -I src/build_parquet.py data/cdc/*.zip`), then run all cells.

**Terms.**
- A *birth cohort* is all babies born in one calendar year. We follow each baby for 365 days.
- The *infant mortality rate* is infant deaths per 1,000 live births.
- *Residents* are births to mothers who live in the U.S. (`RESTATUS` 1–3). CDC tables exclude foreign residents (`RESTATUS` 4).

## 1. Setup
Load the libraries and set the paths. `duckdb` runs SQL directly on the Parquet files, so we never load 30 million rows into memory.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent            # notebooks/ -> InfantMortalityThesis/
sys.path.insert(0, str(ROOT / "src"))

import duckdb
import polars as pl
pl.Config.set_tbl_rows(60); pl.Config.set_tbl_cols(20); pl.Config.set_fmt_str_lengths(40)
PARQUET = ROOT / "data" / "parquet"
COHORTS = range(2016, 2024)
from benchmark import compare, guide_record_counts

## 2. Row counts per cohort

**What.** Count all births, resident births and linked deaths in each Parquet file. Compare them with the record counts that each CDC guide prints.

**Why.** If a row was lost or duplicated while reading the ZIP, the counts will differ.

**Expected.** Our counts equal the guide counts. Most guides count all records, including foreign residents; the 2016 guide counts residents only.

In [2]:
rows = []
for c in COHORTS:
    f = PARQUET / f"cohort_{c}.parquet"
    r = duckdb.sql(f"""SELECT COUNT(*) AS births_all,
                              SUM((CAST(RESTATUS AS INT) < 4)::INT)::BIGINT AS births_res,
                              SUM(death::INT)::BIGINT AS deaths_all,
                              SUM((death AND CAST(RESTATUS AS INT) < 4)::INT)::BIGINT AS deaths_res
                       FROM '{f}'""").fetchall()[0]
    rows.append({"cohort": c, "births_all": r[0], "births_res": r[1], "deaths_all": r[2], "deaths_res": r[3],
                 **guide_record_counts(c)})
counts = pl.DataFrame(rows).with_columns(
    births_match=(pl.col("guide_births") == pl.col("births_all")) | (pl.col("guide_births") == pl.col("births_res")),
    deaths_match=(pl.col("guide_deaths") == pl.col("deaths_all")) | (pl.col("guide_deaths") == pl.col("deaths_res")),
    imr_res=(1000 * pl.col("deaths_res") / pl.col("births_res")).round(2),
)
counts.write_csv(ROOT / "reference" / "benchmark_counts.csv")
counts

cohort,births_all,births_res,deaths_all,deaths_res,guide_deaths,guide_births,births_match,deaths_match,imr_res
i64,i64,i64,i64,i64,i64,i64,bool,bool,f64
2016,3956112,3945875,22927,22893,22893,3945875,true,true,5.8
2017,3864754,3855500,22197,22167,22197,3864754,true,true,5.75
2018,3801534,3791712,21379,21346,21379,3801534,true,true,5.63
2019,3757582,3747540,20681,20639,20681,3757582,true,true,5.51
2020,3619826,3613647,19376,19346,19376,3549826,false,true,5.35
2021,3669928,3664292,19982,19965,19965,3669928,true,true,5.45
2022,3676029,3667758,20299,20268,20299,3676029,true,true,5.53
2023,3605081,3596017,19772,19743,19772,3605081,true,true,5.49


**Result.** Births and deaths match the guide in every cohort, except the births count printed in the 2020 guide (3,549,826). Our file has 3,619,826 records. The difference is exactly 70,000, and NCHS reports 3,613,647 resident births for 2020, which equals our `births_res`. So the 2020 guide has a typing error; the data file is correct.

Deaths match in all 8 cohorts, and no death is left without its birth record (checked when the files were built).

## 3. CDC Documentation Table 2: race × sex × birth weight

**What.** Each guide prints a cohort table of births and infant deaths by mother's race/Hispanic origin, infant sex and birth weight (10 columns). We read that table from the PDF and compute the same table from our data.

**Method detail.** Births come from our Parquet files. Deaths come from the death (numerator) records, because CDC tabulates deaths by the birth weight on the death record. CDC fills in (imputes) missing birth weight separately on the two records, so a few deaths have a different birth weight on each. With the death-record weight, the deaths match exactly.

**Why.** This checks four things together: the race field (`MRACEHISP`), sex (`SEX`), birth weight (`BRTHWGT`) and the death linkage.

**Pre-set check (plan, section 5).** Fail if any rate differs by more than 0.01 per 1,000. We also require exact equality of counts.

In [3]:
tables = pl.concat([compare(c) for c in COHORTS])
tables.write_csv(ROOT / "reference" / "benchmark_table2.csv")
summary = tables.group_by("cohort").agg(
    cells=pl.len(),
    birth_cells_off=(pl.col("diff_births") != 0).sum(),
    death_cells_off=(pl.col("diff_deaths") != 0).sum(),
    total_births_off=pl.col("diff_births").filter(pl.col("bw") == "Total").abs().sum(),
    max_rate_diff=(pl.col("rate") - pl.col("cdc_rate")).abs().max(),
).sort("cohort").with_columns(
    check=pl.when((pl.col("birth_cells_off") == 0) & (pl.col("death_cells_off") == 0) & (pl.col("max_rate_diff") <= 0.01))
           .then(pl.lit("PASS")).otherwise(pl.lit("FAIL")))
summary

cohort,cells,birth_cells_off,death_cells_off,total_births_off,max_rate_diff,check
i32,u32,u32,u32,i64,f64,str
2016,210,150,0,0,300.0,"""FAIL"""
2017,210,0,30,0,1869.53,"""FAIL"""
2018,210,0,0,0,0.0,"""PASS"""
2019,210,0,0,0,0.0,"""PASS"""
2020,210,0,0,0,0.0,"""PASS"""
2021,210,0,0,0,0.0,"""PASS"""
2022,210,0,0,0,0.0,"""PASS"""
2023,210,0,0,0,0.0,"""PASS"""


**Result.** Six cohorts (2018–2023) PASS: all 210 cells are identical. Two cohorts FAIL the strict rule, and both failures come from the CDC tables, not from our data:

- **2016, births only:** 2,493 births are "Not stated" in the CDC table but have a birth weight in our file. All other differences are these same births, now in a weight group, and every total is exact. Our 2016 births file is a re-release (`VS16LINK.DENOMUS_2026_06_04`), newer than the 2016 guide (2019). The likely cause is that the re-release imputes more missing birth weights. Deaths match exactly.
- **2017, NH White deaths only:** the CDC table prints the *all-races* deaths in the NH White rows (for example, 22,167 total deaths). That is a copying error in the guide. All other 180 cells match exactly.

So the data are correct. For the thesis, note that 2016 birth weight differs slightly from the 2016 guide; birth weight is not an input in Part 1 or Part 2.

Below is the "All races, both sexes" part for 2023, as an example of the comparison.

The first block below shows the 2017 NH White error: the CDC deaths equal the all-races deaths.

In [4]:
tables.filter((pl.col("cohort") == 2017) & (pl.col("sex") == "Both") & (pl.col("bw") == "Total")) \
      .select("race", "cdc_deaths", "deaths").sort("race")

race,cdc_deaths,deaths
str,i64,i64
"""All""",22167,22167
"""Hispanic""",4519,4519
"""NH AIAN""",275,275
"""NH Asian""",913,913
"""NH Black""",6085,6085
"""NH NHOPI""",62,62
"""NH White""",22167,9294


In [5]:
tables.filter((pl.col("cohort") == 2023) & (pl.col("race") == "All") & (pl.col("sex") == "Both")) \
      .select("bw", "cdc_births", "births", "cdc_deaths", "deaths", "cdc_rate", "rate")

bw,cdc_births,births,cdc_deaths,deaths,cdc_rate,rate
str,i64,i64,i64,i64,f64,f64
"""Not stated""",948,948,147,147,155.06,155.06
"""<500""",5508,5508,4386,4386,796.3,796.3
"""750-999""",9569,9569,1111,1111,116.1,116.1
"""2000-2499""",199926,199926,1795,1795,8.98,8.98
"""2500+""",3285678,3285678,6701,6701,2.04,2.04
"""500-749""",8427,8427,2811,2811,333.57,333.57
"""1500-1999""",59666,59666,1432,1432,24.0,24.0
"""1250-1499""",14867,14867,645,645,43.38,43.38
"""Total""",3596017,3596017,19743,19743,5.49,5.49


## Summary

- **Main finding.** The Parquet files reproduce CDC's published counts: births and deaths per cohort in all 8 cohorts, and all 210 cells of Documentation Table 2 in 2018–2023.
- **CDC guide errors found.** The 2020 guide prints 3,549,826 records (file and NCHS: 3,619,826). The 2017 table repeats all-races deaths in the NH White rows.
- **One real data difference.** The re-released 2016 births file has birth weight for 2,493 births that the 2016 guide lists as "Not stated". Totals are exact.
- **Open issues.** Table 2 does not check the maternal risk-factor fields; `02_table1` shows their distributions.
- **Next notebook.** `02_table1.ipynb`.